https://www.kaggle.com/c/bike-sharing-demand/data?select=train.csv

In [129]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

이 셀은 데이터 분석 및 머신러닝 모델 구축에 필요한 주요 라이브러리들을 임포트합니다. `sklearn`을 통해 모델 선택, 선형 회귀, 평가 지표를 사용하고, `numpy`는 수치 계산, `matplotlib`는 시각화, `pandas`는 데이터 처리를 담당합니다. `%matplotlib inline`은 주피터 노트북에서 그림을 바로 표시하기 위한 설정입니다.

In [130]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\githome\hipython_rep


데이터 파일의 경로를 설정하기 위해 현재 작업 디렉토리를 변경하는 코드입니다. `os.chdir()` 함수를 사용하여 지정된 경로로 이동하며, `os.getcwd()`를 통해 변경된 디렉토리를 확인합니다.

In [3]:
df = pd.read_csv('data1/bike-sharing-demand/train.csv')

캐글에서 제공하는 자전거 대여량 예측 데이터셋인 'train.csv' 파일을 Pandas DataFrame으로 로드합니다. 이 데이터는 자전거 대여량 예측 모델 구축의 기반이 됩니다.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10886 non-null  object 
 1   season      10886 non-null  int64  
 2   holiday     10886 non-null  int64  
 3   workingday  10886 non-null  int64  
 4   weather     10886 non-null  int64  
 5   temp        10886 non-null  float64
 6   atemp       10886 non-null  float64
 7   humidity    10886 non-null  int64  
 8   windspeed   10886 non-null  float64
 9   casual      10886 non-null  int64  
 10  registered  10886 non-null  int64  
 11  count       10886 non-null  int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 1020.7+ KB


로드된 데이터프레임 `df`의 기본 정보를 확인합니다. 각 컬럼의 이름, Non-Null 값의 개수, 데이터 타입(Dtype), 그리고 메모리 사용량을 통해 데이터의 구조와 누락된 값 여부를 빠르게 파악할 수 있습니다.

In [64]:
dropped_df = df.drop(['datetime','casual','registered'], axis=1)
dropped_df

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,count
0,1,0,0,1,9.84,14.395,81,0.0000,16
1,1,0,0,1,9.02,13.635,80,0.0000,40
2,1,0,0,1,9.02,13.635,80,0.0000,32
3,1,0,0,1,9.84,14.395,75,0.0000,13
4,1,0,0,1,9.84,14.395,75,0.0000,1
...,...,...,...,...,...,...,...,...,...
10881,4,0,1,1,15.58,19.695,50,26.0027,336
10882,4,0,1,1,14.76,17.425,57,15.0013,241
10883,4,0,1,1,13.94,15.910,61,15.0013,168
10884,4,0,1,1,13.94,17.425,61,6.0032,129


모델 학습에 직접적으로 사용되지 않거나, `count` 컬럼과 중복 정보를 가지는 'datetime', 'casual', 'registered' 컬럼을 데이터프레임에서 제거합니다. 이는 모델의 복잡성을 줄이고 예측 성능을 향상시키기 위함입니다. `dropped_df`를 출력하여 변경된 데이터프레임의 모습을 확인합니다.

In [65]:
X = dropped_df.drop('count',axis=1).values
y = dropped_df['count']

예측하고자 하는 'count' 컬럼을 타겟 변수 `y`로 설정하고, 'count'를 제외한 나머지 모든 컬럼들을 피처(독립 변수) `X`로 분리합니다. 이는 머신러닝 모델 학습을 위한 표준적인 데이터 준비 과정입니다.

In [66]:
from sklearn.preprocessing import StandardScaler

데이터 스케일링을 위해 `sklearn.preprocessing` 모듈에서 `StandardScaler`를 임포트합니다. 이는 피처들의 스케일이 다를 때 모델 성능에 미칠 수 있는 부정적인 영향을 줄여줍니다.

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

원본 데이터를 훈련 세트와 테스트 세트로 8:2 비율로 분리합니다. `random_state=50`은 재현 가능한 결과를 보장합니다. 이후 `StandardScaler`를 사용하여 훈련 세트(`X_train`)에 맞춰 스케일링을 학습하고, 이 스케일링을 훈련 및 테스트 세트(`X_train_scaled`, `X_test_scaled`)에 적용합니다.

In [68]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred = lr.predict(X_test_scaled)

이 셀은 **선형 회귀 모델(`LinearRegression`)을 초기화하고, 스케일링된 훈련 데이터(`X_train_scaled`, `y_train`)를 사용하여 모델을 학습**시킵니다. 학습된 모델은 `fit()` 메서드를 통해 데이터의 패턴을 파악하고, `predict()` 메서드를 사용하여 스케일링된 테스트 데이터(`X_test_scaled`)에 대한 자전거 대여량 예측값(`y_pred`)을 생성합니다. 이 예측값은 이후 모델의 성능을 평가하는 데 사용됩니다.

In [69]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(155.13350586282309), np.float64(24066.404641290563))

이 셀은 선형 회귀 모델의 **예측 성능을 평가하기 위해 평균 제곱 오차(MSE)와 제곱근 평균 제곱 오차(RMSE)를 계산**합니다. `mean_squared_error` 함수는 실제 값(`y_test`)과 예측 값(`y_pred`) 간의 오차 제곱 평균을 구하고, `np.sqrt()`를 통해 MSE의 제곱근인 RMSE를 계산합니다. RMSE는 예측 오류의 크기를 실제 값과 동일한 단위로 나타내므로 모델의 예측 정확도를 직관적으로 이해하는 데 유용합니다.

In [70]:
r2_score(y_test, y_pred)

np.float64(0.2613088601741649)

이 셀은 선형 회귀 모델의 **결정 계수(R2 스코어)를 계산하여 모델의 설명력을 평가**합니다. `r2_score` 함수는 모델이 종속 변수(`y_test`)의 분산을 얼마나 잘 설명하는지를 0과 1 사이의 값으로 나타냅니다. 1에 가까울수록 모델이 데이터를 잘 설명하며 예측력이 높다고 볼 수 있습니다.

In [71]:
pd.Series(data = np.abs(np.round(lr.coef_,1)), index=dropped_df.drop('count',axis=1).columns).sort_values(ascending=False)

humidity      58.4
atemp         47.1
season        26.0
temp          14.8
windspeed      7.4
weather        3.4
holiday        0.8
workingday     0.4
dtype: float64

이 셀은 **선형 회귀 모델의 회귀 계수(`lr.coef_`)를 활용하여 각 피처의 중요도를 분석**합니다. 회귀 계수의 절댓값을 사용하여 피처가 타겟 변수(`count`)에 미치는 영향의 크기를 파악하고, 이를 Pandas Series로 변환한 후 내림차순으로 정렬합니다. 이 결과를 통해 어떤 피처(`humidity`, `atemp`, `season` 등)가 자전거 대여량 예측에 가장 큰 영향을 미치는지 직관적으로 이해할 수 있습니다. 이는 모델 해석 및 피처 엔지니어링 전략 수립에 중요한 통찰력을 제공합니다.

In [77]:
X = dropped_df.drop(['count', 'workingday', 'holiday'],axis=1).values
y = dropped_df['count']

이 셀은 **기존 `dropped_df`에서 'count' 뿐만 아니라 'workingday'와 'holiday' 컬럼을 추가로 제외하여 피처 `X`를 재설정**합니다. 이는 앞서 수행한 피처 중요도 분석 결과('workingday', 'holiday'의 중요도가 낮음)를 바탕으로 모델의 불필요한 복잡성을 줄이고 성능 개선을 시도하기 위함입니다. 타겟 변수 `y`는 동일하게 'count' 컬럼으로 유지됩니다.

In [115]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

이 셀은 **재설정된 피처 `X`와 타겟 `y`를 사용하여 데이터를 다시 훈련 세트와 테스트 세트로 분리하고 스케일링**합니다. `train_test_split`을 통해 8:2 비율로 분리하며 `random_state=50`을 유지하여 일관성을 확보합니다. 이후 `StandardScaler`를 다시 `fit`하고 `transform`하여 새로운 피처 조합에 맞춰 데이터를 표준화합니다. 이 과정은 변경된 피처 셋으로 모델을 재학습하기 위한 필수적인 데이터 전처리입니다.

In [79]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred = lr.predict(X_test_scaled)

이 셀은 **재조정된 데이터(`X_train_scaled`, `y_train`)를 사용하여 선형 회귀 모델을 다시 학습**하고 예측을 수행합니다. 피처에서 'workingday'와 'holiday'가 제외되었으므로, 이전 모델과는 다른 학습 결과를 보일 수 있습니다. 이 단계를 통해 피처 선택이 모델 성능에 미치는 영향을 평가할 수 있습니다.

In [80]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(155.15284956651055), np.float64(24072.406728608254))

이 셀은 피처가 조정된 후 **재학습된 선형 회귀 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 다시 계산**합니다. 이전 모델의 평가 결과와 비교하여, 'workingday'와 'holiday' 컬럼 제거가 모델의 오차 크기에 어떤 영향을 미쳤는지 확인할 수 있습니다.

In [82]:
r2_score(y_test, y_pred)

np.float64(0.26112463287523413)

In [83]:
from sklearn.model_selection import cross_val_score
neg_mse_scores = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_scores

array([-13804.19466134, -25029.44587499, -16211.79111405, -40569.56932921,
       -45529.14186029])

In [84]:
RMSE = np.sqrt(neg_mse_scores * (-1))
np.mean(RMSE), RMSE

(np.float64(163.5636472332233),
 array([117.49125355, 158.20697164, 127.32553206, 201.4188902 ,
        213.37558872]))

In [85]:
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([-0.39292789, -0.11164959,  0.02454896,  0.12903436,  0.03832124]),
 np.float64(-0.06253458470757459))

In [88]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=42, max_depth=8)
rf.fit(X_train_scaled,y_train)
y_pred = rf.predict(X_test_scaled)

In [89]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(147.89997628637894), np.float64(21874.402985511457))

In [90]:
r2_score(y_test, y_pred)

np.float64(0.3285898780803259)

In [91]:
neg_mse_scores = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_scores

array([-13560.09864193, -26769.58352183, -18604.0141961 , -42910.36686374,
       -43370.70482081])

In [92]:
RMSE = np.sqrt(neg_mse_scores * (-1))
np.mean(RMSE), RMSE

(np.float64(166.3726039154419),
 array([116.44783657, 163.61412996, 136.39653293, 207.14817611,
        208.25634401]))

In [93]:
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([-0.36829711, -0.1889355 , -0.11938927,  0.07878108,  0.08391233]),
 np.float64(-0.10278569404440611))

현재까지 랜덤포레스트 단독 모델이 가장 낫다

train_test_split의 random_state가 50일때

| 모델         | 평가 지표 | 단독 예측 (Individual Prediction) | 교차 검증 (Cross-Validation) |
| :----------- | :-------- | :-------------------------------- | :--------------------------- |
| **리니어 모델** | RMSE      | 155.15                            | 163.56                       |
|              | R2        | 0.2611                            | -0.0625                      |
| **랜덤 포레스트** | RMSE      | 147.89                            | 166.37                       |
|              | R2        | 0.3286                            | -0.1028                      |

In [94]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

In [96]:
results = []
for degree in range(1,6):
    model_poly = Pipeline([
        ('poly',PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear',LinearRegression())
    ])
    model_poly.fit(X_train_scaled, y_train)
    pred_poly = model_poly.predict(X_test_scaled)
    mse = mean_squared_error(y_test,pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({
        'degree' : degree,
        'MSE' : mse,
        'RMSE' : rmse,
        'R2' : r2
    })
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,1,2.407241e+04,155.152850,0.261125
1,2,2.338400e+04,152.918263,0.282255
2,3,2.285258e+04,151.170706,0.298566
3,4,2.258853e+04,150.294799,0.306671
4,5,2.443706e+04,156.323580,0.249932
5,6,8.307877e+06,2882.338837,-254.000917


다항회귀모델(선형)에서는 4차가 가장 높다

In [116]:
results = []
best_r2 = False
for degree in range(1,6):
    model_poly = Pipeline([
        ('poly',PolynomialFeatures(degree=degree, include_bias=False)),
        ('RF',RandomForestRegressor(random_state=22, max_depth=8))
    ])
    model_poly.fit(X_train_scaled, y_train)
    pred_poly = model_poly.predict(X_test_scaled)
    mse = mean_squared_error(y_test,pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({
        'degree' : degree,
        'MSE' : mse,
        'RMSE' : rmse,
        'R2' : r2
    })
    if best_r2 < r2 or best_r2 == False:
        best_r2 = r2
        best_model = model_poly
        best_pred_poly = pred_poly
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,1,21939.989328,148.121536,0.326577
1,2,21686.767855,147.264279,0.334349
2,3,21951.341017,148.159850,0.326228
3,4,21972.965541,148.232809,0.325565
4,5,22010.061124,148.357882,0.324426


다항회귀모델(랜덤포레스트)에서는 2차가 가장 높다

결론적으로 랜덤포레스트 랜덤포레스트 다항회귀모델일때 성능이 가장 높았다.

In [30]:
from sklearn.ensemble import GradientBoostingRegressor

In [98]:
gb_clf = GradientBoostingRegressor()
gb_clf.fit(X_train_scaled,y_train)
gb_pred = gb_clf.predict(X_test_scaled)

In [99]:
mse = mean_squared_error(y_test,gb_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(149.49095795412498), np.float64(22347.54651004196))

In [100]:
r2_score(y_test, gb_pred)

np.float64(0.3140672713741707)

In [101]:
from xgboost import XGBRegressor

In [102]:
xgb = XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)
evals = [(X_test_scaled, y_test)]
xgb.fit(X_train_scaled, y_train, early_stopping_rounds=40, 
        eval_set=evals, verbose=True)
xgb_pred = xgb.predict(X_test_scaled)

[0]	validation_0-rmse:242.84177
[1]	validation_0-rmse:228.61965
[2]	validation_0-rmse:216.49434
[3]	validation_0-rmse:206.01087
[4]	validation_0-rmse:197.15921
[5]	validation_0-rmse:189.59136
[6]	validation_0-rmse:183.03899
[7]	validation_0-rmse:177.56206
[8]	validation_0-rmse:173.11372
[9]	validation_0-rmse:169.33266
[10]	validation_0-rmse:166.15398
[11]	validation_0-rmse:163.56794
[12]	validation_0-rmse:161.43890
[13]	validation_0-rmse:159.57331
[14]	validation_0-rmse:158.08862
[15]	validation_0-rmse:156.90389
[16]	validation_0-rmse:155.85758
[17]	validation_0-rmse:154.95150
[18]	validation_0-rmse:154.20779
[19]	validation_0-rmse:153.59687
[20]	validation_0-rmse:153.15421
[21]	validation_0-rmse:152.67138
[22]	validation_0-rmse:152.27149
[23]	validation_0-rmse:151.99234
[24]	validation_0-rmse:151.70967
[25]	validation_0-rmse:151.46073
[26]	validation_0-rmse:151.33540
[27]	validation_0-rmse:151.20008
[28]	validation_0-rmse:151.02382
[29]	validation_0-rmse:150.88658
[30]	validation_0-rm

c:\Users\Admin\miniconda3\envs\hi_ml_env\lib\site-packages\xgboost\sklearn.py:793: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[141]	validation_0-rmse:149.50530


In [105]:
mse = mean_squared_error(y_test,xgb_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(149.4188317499895), np.float64(22325.98728153167))

In [104]:
r2_score(y_test, xgb_pred)

np.float64(0.3147290075711392)

In [47]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

In [117]:
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled,y_train)
pred_ridge = ridge.predict(X_test_scaled)

mse = mean_squared_error(y_test, pred_ridge)
r2 = r2_score(y_test, pred_ridge)
mse, r2

(np.float64(24072.976546867623), np.float64(0.2611071429466063))

In [118]:
from sklearn.linear_model import RidgeCV, LassoCV
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
ridge_preds = ridge_cv.predict(X_test_scaled)
ridge_mse = mean_squared_error(y_test, ridge_preds)
ridge_r2 = r2_score(y_test, ridge_preds)
print(f'ridge cv mse : {ridge_mse:.4f}, r2 : {ridge_r2:.4f}')

ridge cv mse : 24072.9765, r2 : 0.2611


In [119]:
ridge_cv.alpha_

np.float64(10.0)

In [120]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled,y_train)
pred_lasso = lasso.predict(X_test_scaled)

mse = mean_squared_error(y_test, pred_lasso)
r2 = r2_score(y_test, pred_lasso)
mse, r2

(np.float64(24072.06180558391), np.float64(0.26113521989833965))

In [121]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
lasso_cv = LassoCV(alphas=alphas, cv=5)
lasso_cv.fit(X_train_scaled, y_train)
lasso_preds = lasso_cv.predict(X_test_scaled)
lasso_mse = mean_squared_error(y_test, lasso_preds)
lasso_r2 = r2_score(y_test, lasso_preds)
print(f'ridge cv mse : {lasso_mse:.4f}, r2 : {lasso_r2:.4f}')

ridge cv mse : 24072.4124, r2 : 0.2611


In [122]:
lasso_cv.alpha_

np.float64(0.001)

In [123]:
lasso_cv.coef_

array([ 25.96650791,   3.39270523,  14.6792473 ,  47.21051443,
       -58.36703917,   7.43558374])

In [124]:
ridge_cv.coef_

array([ 25.9282778 ,   3.34483943,  15.86486109,  46.00117238,
       -58.25891329,   7.40861484])

In [125]:
enet = ElasticNet(alpha=0.1, l1_ratio=0.5)
enet.fit(X_train_scaled,y_train)

ElasticNet(alpha=0.1)

In [126]:
enet_pred = enet.predict(X_test_scaled)

In [127]:
mse = mean_squared_error(y_test, enet_pred)
r2 = r2_score(y_test, enet_pred)
mse, r2

(np.float64(24103.33163284492), np.float64(0.26017542782778))

In [128]:
results = pd.DataFrame({
    '모델' : ['다항회귀', '릿지회귀', '라쏘회귀', '엘라스틱넷회귀'],
    'RMSE' : [np.sqrt(mean_squared_error(y_test,best_pred_poly)),
             np.sqrt(mean_squared_error(y_test,pred_ridge)),
             np.sqrt(mean_squared_error(y_test,pred_lasso)),
             np.sqrt(mean_squared_error(y_test,enet_pred))],
    'R2' : [r2_score(y_test,best_pred_poly),
            r2_score(y_test,pred_ridge),
            r2_score(y_test,pred_lasso),
            r2_score(y_test,enet_pred)]
})
results

,모델,RMSE,R2
0,다항회귀,147.264279,0.334349
1,릿지회귀,155.154686,0.261107
2,라쏘회귀,155.151738,0.261135
3,엘라스틱넷회귀,155.252477,0.260175
